# AGRIVISION Model 2 - Plant Health / Crop Analysis

Trains the **second** model in the AGRIVISION pipeline. Model 1 (COCO-SSD, in the
browser) answers *"is there a plant?"*. This model answers *"what crop, is it
healthy, and what condition does it show?"* for each plant Model 1 found.

**Runtime > Change runtime type > GPU (T4)**, then **Runtime > Run all**.

---

## Read this before trusting any accuracy number

The training data is **PlantVillage**: 54,303 lab photographs of single detached
leaves on uniform backgrounds. Noyan (2022, [arXiv:2206.04374](https://arxiv.org/abs/2206.04374))
trained a classifier on **8 background pixels alone** and reached **49.0%**
accuracy against a **2.6%** random baseline - the backgrounds leak the label.
PlantVillage-trained models measured at ~99% in-domain have been reported at
**~31%** on other datasets.

So this notebook evaluates on **two** sets:

| set | what it is | what it tells you |
|---|---|---|
| PlantVillage test split | held-out lab images | almost nothing about the real world |
| **PlantDoc** | field photographs, never trained on | **the number that matters** |

`evaluate.py` prints both and flags the gap. Report the PlantDoc number.

In [ ]:
# 1. Get the training code. Point this at your own repo, or upload the folder.
REPO_URL = ""   # e.g. "https://github.com/<you>/cv-prototype.git"

import os, sys, shutil, subprocess
from pathlib import Path

if REPO_URL:
    if not Path("cv-prototype").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "cv-prototype"], check=True)
    PH = Path("cv-prototype/training/plant_health").resolve()
else:
    # Fallback: upload a zip of the training/plant_health folder.
    from google.colab import files
    if not Path("plant_health").exists():
        print("Upload a zip of training/plant_health ...")
        up = files.upload()
        name = next(iter(up))
        shutil.unpack_archive(name, ".")
    PH = Path("plant_health").resolve()

assert (PH / "config.py").exists(), f"config.py not found under {PH}"
os.chdir(PH)
sys.path.insert(0, str(PH))
print("Working dir:", Path.cwd())

In [ ]:
# 2. Dependencies. Colab already has TensorFlow; we add TFDS + sklearn helpers.
!pip install -q tensorflow_datasets scikit-learn matplotlib pillow

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "NONE - switch Runtime to GPU!")

## 3. Prepare the dataset

Downloads PlantVillage (via TFDS - no Kaggle login needed) and PlantDoc, then:
drops corrupt/blank images, collapses visual duplicates with a difference hash,
and splits **by duplicate cluster** so the same physical leaf can never appear in
both train and test. Class-level pruning removes anything below
`MIN_IMAGES_PER_CLASS`.

First run downloads ~2 GB and takes 10-20 minutes.

In [ ]:
!python scripts/prepare_dataset.py

import json
from pathlib import Path
m = json.loads(Path("dataset/manifest.json").read_text())
print(f"\nclasses: {m['num_classes']}  crops: {m['crops']}")
print(f"healthy/unhealthy images: {m['healthy_images']} / {m['unhealthy_images']}")
print(f"OOD (PlantDoc) images: {m['ood']['images']}")

## 4. Train

MobileNetV3-Small, two phases: frozen backbone (LR 1e-3), then fine-tune the top
30% of the backbone (LR 1e-5). Class weights handle imbalance; BatchNorm stays
frozen during fine-tuning.

Add `--quick` to smoke-test the pipeline in ~2 minutes before committing to a
full run.

In [ ]:
!python scripts/train.py

## 5. Evaluate - the honest part

Prints accuracy, macro/weighted precision, recall and F1, per-class tables and
confusion matrices for **both** sets, plus the health-level binary view.

Pay attention to **MISSED DISEASE** (an unhealthy plant called healthy). That is
the operationally dangerous error: a false alarm costs a farmer an inspection, a
missed disease costs a crop.

In [ ]:
!python scripts/evaluate.py

import json
from pathlib import Path
r = json.loads(Path("models/reports/evaluation.json").read_text())
ind, ood = r["in_domain_test_plantvillage"], r["out_of_domain_plantdoc"]
if ind: print(f"in-domain  : {ind['accuracy']:.1%}")
if ood: print(f"OUT-OF-DOMAIN: {ood['accuracy']:.1%}  <-- report THIS one")

In [ ]:
# Confusion matrices
from IPython.display import Image, display
from pathlib import Path
for p in sorted(Path("models/reports").glob("*_confusion.png")):
    print(p.name); display(Image(str(p)))

## 6. Export to TensorFlow Lite

Converts the best checkpoint and then **verifies** it by comparing the TFLite
interpreter against Keras on real test images (top-1 agreement + probability
drift). A silent behaviour change during conversion is exactly how Model 1's
first version shipped broken, so this check gates the export.

Use `--int8` for a smaller model if inference is too slow on the rover; expect a
small accuracy cost and re-check the agreement number.

In [ ]:
!python scripts/export_tflite.py

In [ ]:
# 7. Download the two files to drop into cv-prototype/model/
import shutil
from google.colab import files
shutil.make_archive("plant_health_model", "zip", "models/export")
files.download("plant_health_model.zip")

## 8. Install into the AGRIVISION app

Unzip and copy **both** files into `cv-prototype/model/`:

```
plant_health_classifier.tflite
class_names.json
```

Reload the page. `script.js` detects them and enables the analysis stage
automatically; if they are absent, detection/counting/popup all keep working
exactly as before.

**Before believing the app:** hold up a plant and check that the reported crop is
right. If the model was trained only on PlantVillage, expect it to struggle on a
whole potted plant in a room - it was trained on single detached leaves against
a plain background. Feeding it Model 1's crop of a *whole plant* is already a
domain shift on top of the lab-to-field gap. The fix is to add field imagery to
the training set (PlantDoc's train split, or your own rover captures), not to
lower the confidence thresholds.